# 01 — Data Preprocessing

**CognitiveCyber** — An Open Cognitive Cybersecurity Research Platform

This notebook loads a network-flow intrusion-detection dataset, auto-detects
its schema, cleans/encodes/scales it, and produces the train/val/test splits
used by every downstream notebook.

**Data note:** CICIDS2017 / CSE-CIC-IDS2018 / UNSW-NB15 / TON_IoT / Bot-IoT
are multi-gigabyte files hosted by third-party institutions and are not
reachable from this notebook's execution environment. By default this
notebook generates a **synthetic, schema-compatible** UNSW-NB15-style flow
dataset (`cognitivecyber.data.synthetic`) so every cell below runs and
produces real, verifiable output. To use a real dataset instead, download
one of the supported CSVs to `datasets/raw/<file>.csv` and change
`DATASET_PATH` below — the rest of the pipeline is schema-agnostic and
requires no other changes.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and Path.cwd().name == "notebooks" else Path.cwd()
if (Path.cwd() / "src").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd

from cognitivecyber.data.loaders import load_dataset
from cognitivecyber.data.preprocessing import clean_and_encode, split_and_scale

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set to a real dataset CSV path (e.g. REPO_ROOT / "datasets" / "raw" / "UNSW_NB15.csv")
# to run against real data instead of the synthetic generator.
DATASET_PATH = None
DATASET_NAME = None  # e.g. "UNSW-NB15"; auto-detected if None

print("Repo root:", REPO_ROOT)

Repo root: /home/claude/CognitiveCyber


## 1. Load dataset + auto-detect schema

In [2]:
df, meta = load_dataset(path=DATASET_PATH, dataset_name=DATASET_NAME, n_synthetic=60_000, random_state=RANDOM_STATE)

print("Shape:", df.shape)
print("Schema detected:", meta["schema_name"], "| synthetic:", meta["is_synthetic"])
print("Label column:", meta["label_column"], "| binary:", meta["is_binary"])
df.head()

Shape: (60000, 22)
Schema detected: synthetic_unsw_style | synthetic: True
Label column: label | binary: True


,dur,sbytes,dbytes,spkts,dpkts,sttl,dttl,sload,dload,swin,...,dmean,ct_srv_src,ct_dst_ltm,ct_state_ttl,proto,service,state,attack_cat,label,is_synthetic
0,0.089268,1706.905105,173.584414,9.0,8.0,63.0,60.0,416.732816,1460.596131,259.0,...,242.471911,2.0,3.0,2.0,tcp,-,REQ,Normal,0,True
1,0.018962,210.565363,127.600574,8.0,6.0,61.0,62.0,1529.552312,2063.078416,252.0,...,259.001154,1.0,1.0,1.0,tcp,ftp,FIN,Normal,0,True
2,0.010209,59836.943660,58224.242455,401.0,136.0,38.0,38.0,2935.003234,2364.253100,211.0,...,158.256365,13.0,10.0,0.0,tcp,dns,FIN,DoS,1,True
3,0.117048,447.313789,696.283440,4.0,7.0,61.0,62.0,1129.672673,1060.000561,258.0,...,318.876537,3.0,4.0,6.0,udp,dns,FIN,Normal,0,True
4,0.190618,467.835773,809.137848,6.0,4.0,62.0,60.0,3367.330282,187.323436,259.0,...,257.030943,2.0,3.0,3.0,tcp,http,CON,Normal,0,True


## 2. Class balance and basic profiling

In [3]:
print("Attack ratio (label==1):", df["label"].mean().round(4))
if "attack_cat" in df.columns:
    display(df["attack_cat"].value_counts())

print("\nMissing values per column:")
display(df.isna().sum()[df.isna().sum() > 0])

print("\nDtypes:")
display(df.dtypes.value_counts())

Attack ratio (label==1): 0.35


attack_cat
Normal            39000
DoS                2625
Exploits           2625
Reconnaissance     2625
PortScan           2625
BruteForce         2625
DDoS               2625
Botnet             2625
Backdoor           2625
Name: count, dtype: int64


Missing values per column:


sload    641
dload    597
smean    560
dmean    556
dtype: int64


Dtypes:


float64    16
str         4
int64       1
bool        1
Name: count, dtype: int64

## 3. Clean, encode categoricals, impute missing values

In [4]:
X, y = clean_and_encode(df, meta)
print("Feature matrix shape after cleaning/encoding:", X.shape)
print("Any remaining NaNs:", X.isna().sum().sum())
print("Any remaining non-numeric columns:", X.select_dtypes(exclude=[np.number]).columns.tolist())
X.head()

Feature matrix shape after cleaning/encoding: (60000, 16)
Any remaining NaNs: 0
Any remaining non-numeric columns: []


,dur,sbytes,dbytes,spkts,dpkts,sttl,dttl,sload,dload,swin,dwin,smean,dmean,ct_srv_src,ct_dst_ltm,ct_state_ttl
0,0.089268,1706.905105,173.584414,9.0,8.0,63.0,60.0,416.732816,1460.596131,259.0,252.0,236.661369,242.471911,2.0,3.0,2.0
1,0.018962,210.565363,127.600574,8.0,6.0,61.0,62.0,1529.552312,2063.078416,252.0,256.0,316.630625,259.001154,1.0,1.0,1.0
2,0.010209,59836.943660,58224.242455,401.0,136.0,38.0,38.0,2935.003234,2364.253100,211.0,20.0,98.242527,158.256365,13.0,10.0,0.0
3,0.117048,447.313789,696.283440,4.0,7.0,61.0,62.0,1129.672673,1060.000561,258.0,251.0,310.103501,318.876537,3.0,4.0,6.0
4,0.190618,467.835773,809.137848,6.0,4.0,62.0,60.0,3367.330282,187.323436,259.0,253.0,342.206893,257.030943,2.0,3.0,3.0


## 4. Stratified train/val/test split + scaling

In [5]:
pdata = split_and_scale(X, y, test_size=0.2, val_size=0.1, random_state=RANDOM_STATE)

print("Train:", pdata.X_train.shape, "| attack ratio:", pdata.y_train.mean().round(4))
print("Val:  ", pdata.X_val.shape,   "| attack ratio:", pdata.y_val.mean().round(4))
print("Test: ", pdata.X_test.shape,  "| attack ratio:", pdata.y_test.mean().round(4))
print("Feature count:", len(pdata.feature_names))

Train:

 (42000, 16) | attack ratio: 0.35
Val:   (6000, 16) | attack ratio: 0.35
Test:  (12000, 16) | attack ratio: 0.35
Feature count: 16


## 5. Persist the cleaned dataset + splits for downstream notebooks

In [6]:
import joblib

out_dir = REPO_ROOT / "datasets" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(REPO_ROOT / "datasets" / "synthetic_flows.csv", index=False)
joblib.dump(pdata, out_dir / "splits.joblib")

print("Saved raw dataset  ->", REPO_ROOT / "datasets" / "synthetic_flows.csv")
print("Saved train/val/test splits + scaler -> ", out_dir / "splits.joblib")

Saved raw dataset  -> /home/claude/CognitiveCyber/datasets/synthetic_flows.csv
Saved train/val/test splits + scaler ->  /home/claude/CognitiveCyber/datasets/processed/splits.joblib


## Summary

- Loaded a schema-compatible network-flow dataset (synthetic by default;
  swap in a real CICIDS2017/UNSW-NB15/TON_IoT/Bot-IoT CSV via `DATASET_PATH`
  with zero pipeline changes).
- Verified class balance, missingness, and dtypes.
- Cleaned, imputed, one-hot encoded, and scaled features.
- Produced stratified train/val/test splits, persisted for notebook `02_Baseline_Models.ipynb`.